# 02f — IAPs (In-App Purchases)

Procesa la monetización: **consumables** (compras únicas como paquetes
de gemas) y **subscriptions** (mensual, trimestral, anual). Crítico para
el modelo de churn — usuarios que pagan churnean menos (efecto sunk-cost,
Hadiji/Periáñez/Bergqvist).

**Inputs:**
- `data/data_raw/processed_consumables_iaps.csv` (56 MB, ~31K filas)
- `data/data_raw/processed_subscriptions_iaps.csv` (37 MB, ~28K filas)
- `data/data_qc/sample_user_ids.parquet`
- `data/data_qc/features_arena.parquet` (para cross-check)

**NO se procesa:** `subscriptions_iap.csv` (507 B, catálogo de productos).

**Outputs:**
- `data/data_qc/features_iaps.parquet` (126.253 × 23 cols)

**Políticas clave:**
- Filas con `user_id` NaN se eliminan (no atribuibles).
- `REFERENCE_DATE` como CUTOFF point-in-time para TODAS las features
  temporales. No se mira ningún dato posterior.
- Mapeo char_id → user_id NO se necesita aquí (los CSVs usan user_id directo).

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'iaps'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_CONSUMABLES = DATA_RAW / 'processed_consumables_iaps.csv'
CSV_SUBSCRIPTIONS = DATA_RAW / 'processed_subscriptions_iaps.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_iaps_cutoff.parquet'

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_CONSUMABLES existe: {CSV_CONSUMABLES.exists()}")
print(f"CSV_SUBSCRIPTIONS existe: {CSV_SUBSCRIPTIONS.exists()}")

PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_CONSUMABLES existe: True
CSV_SUBSCRIPTIONS existe: True


In [3]:
# [SETUP] Carga sample_user_ids, REFERENCE_DATE y ventanas unix
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

# Constantes temporales en unix seconds
REFERENCE_DATE_UNIX = int(REFERENCE_DATE.timestamp())
CUTOFF_DATE_UNIX = int(CUTOFF_DATE.timestamp())
# Ventanas relativas al CUTOFF_DATE (no al REFERENCE_DATE) para preservar
# la semántica temporal: las features describen actividad ANTES del cutoff.
UNIX_7D_AGO  = CUTOFF_DATE_UNIX - 7 * 86400
UNIX_30D_AGO = CUTOFF_DATE_UNIX - 30 * 86400
UNIX_90D_AGO = CUTOFF_DATE_UNIX - 90 * 86400

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
print(f"REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}")
print(f"Ventana 7d:  [{UNIX_7D_AGO}, {CUTOFF_DATE_UNIX}]")
print(f"Ventana 30d: [{UNIX_30D_AGO}, {CUTOFF_DATE_UNIX}]")
print(f"Ventana 90d: [{UNIX_90D_AGO}, {CUTOFF_DATE_UNIX}]")

log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}')

Usuarios en sample: 25,200
REFERENCE_DATE: 2026-04-04 18:23:13
REFERENCE_DATE_UNIX: 1775326993
Ventana 7d:  [1764354193, 1764958993]
Ventana 30d: [1762366993, 1764958993]
Ventana 90d: [1757182993, 1764958993]
[SETUP] 13:01:46 — sample_user_ids: 25,200 usuarios
[SETUP] 13:01:46 — REFERENCE_DATE_UNIX: 1775326993


## 1. Consumables — carga y exploración

In [4]:
# [EXEC] Cargar processed_consumables_iaps.csv
t0 = time.time()
df_cons_raw = pd.read_csv(CSV_CONSUMABLES, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_cons_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
print(f"Memoria: {df_cons_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
log_step('EXEC', f'Carga consumables CSV: shape={df_cons_raw.shape}, tiempo={load_time:.1f}s')

Shape: (31176, 13)
Tiempo: 0.4s
Memoria: 62.8 MB
[EXEC] 13:01:46 — Carga consumables CSV: shape=(31176, 13), tiempo=0.4s


In [5]:
# [ANALYSIS] Exploración consumables — sobre CSV COMPLETO (no muestra)
print("TIPOS DE DATOS — consumables raw")
print("=" * 80)
for col in df_cons_raw.columns:
    dtype = df_cons_raw[col].dtype
    n_nulls = df_cons_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_cons_raw) * 100
    n_unique = df_cons_raw[col].nunique()
    example = df_cons_raw[col].dropna().iloc[0] if n_nulls < len(df_cons_raw) else 'TODO NULO'
    print(f"  {col:<20} dtype={str(dtype):<10} nulls={n_nulls:>8,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ej='{str(example)[:30]}'")

# Nulos por columna → CSV
nulls_cons = df_cons_raw.isnull().sum().to_frame(name='n_nulls')
nulls_cons['pct_nulls'] = (nulls_cons['n_nulls'] / len(df_cons_raw) * 100).round(2)
nulls_cons = nulls_cons.sort_values('pct_nulls', ascending=False)
nulls_cons.to_csv(REPORT_DIR / 'nulos_por_columna_raw_consumables.csv')

# Distribuciones clave
print("\n=== platform ===")
print(df_cons_raw['platform'].value_counts(dropna=False).head(10))

print("\n=== product_id (top 20) ===")
print(df_cons_raw['product_id'].value_counts(dropna=False).head(20))

print("\n=== purchase_state ===")
print(df_cons_raw['purchase_state'].value_counts(dropna=False))

print("\n=== purchase_time ===")
pt_min = df_cons_raw['purchase_time'].min()
pt_max = df_cons_raw['purchase_time'].max()
pt_med = df_cons_raw['purchase_time'].median()
print(f"  min: {pt_min} ({pd.to_datetime(pt_min, unit='s')})")
print(f"  max: {pt_max} ({pd.to_datetime(pt_max, unit='s')})")
print(f"  med: {pt_med:.0f} ({pd.to_datetime(pt_med, unit='s')})")

# % user_id NaN sobre CSV COMPLETO
pct_uid_nan_cons = df_cons_raw['user_id'].isna().mean() * 100
print(f"\n% user_id NaN en consumables (CSV completo): {pct_uid_nan_cons:.2f}%")
log_step('ANALYSIS', f'Consumables: {pct_uid_nan_cons:.2f}% user_id NaN (CSV completo)')

TIPOS DE DATOS — consumables raw
  _id                  dtype=str        nulls=       0 (  0.0%)  unique=  31,176  ej='ObjectId(5fc50e1b88f10713eb27d'
  transaction_id       dtype=str        nulls=       0 (  0.0%)  unique=  31,176  ej='1000000748142987'
  platform             dtype=str        nulls=       0 (  0.0%)  unique=       2  ej='ios'
  store                dtype=str        nulls=   5,455 ( 17.5%)  unique=       5  ej='xsolla'
  product_id           dtype=str        nulls=       0 (  0.0%)  unique=      13  ej='gems2'
  user_id              dtype=str        nulls=  11,129 ( 35.7%)  unique=   6,342  ej='5fe70885cf4f3a29454a003a'
  purchase_time        dtype=int64      nulls=       0 (  0.0%)  unique=  31,171  ej='1606748648'
  purchase_state       dtype=float64    nulls=      90 (  0.3%)  unique=       2  ej='1.0'
  purchase_token       dtype=str        nulls=   4,978 ( 16.0%)  unique=  26,070  ej='hbfldjbbbnojlolgbanfmphf.AO-J1'
  reviewed             dtype=object     nulls=  

In [6]:
# [EXEC] Filtrar consumables
n_before_cons = len(df_cons_raw)

# 1. Eliminar user_id NaN (no atribuibles)
df_cons = df_cons_raw.dropna(subset=['user_id']).copy()
n_no_nan_cons = len(df_cons)

# 2. Normalizar user_id
df_cons['user_id'] = df_cons['user_id'].apply(clean_user_id)

# 3. Filtrar por sample_user_ids
df_cons = df_cons[df_cons['user_id'].isin(sample_user_ids)].copy()
n_in_sample_cons = len(df_cons)

# 4. Convertir purchase_time a datetime
df_cons['purchase_dt'] = pd.to_datetime(df_cons['purchase_time'], unit='s', errors='coerce')

print(f"Filas iniciales:           {n_before_cons:>10,}")
print(f"Tras eliminar user_id NaN: {n_no_nan_cons:>10,}  ({n_no_nan_cons/n_before_cons*100:.1f}%)")
print(f"Tras filtro sample:        {n_in_sample_cons:>10,}  ({n_in_sample_cons/n_before_cons*100:.1f}%)")

n_users_with_cons = df_cons['user_id'].nunique()
print(f"\nUsuarios con consumables: {n_users_with_cons:,} ({n_users_with_cons/N_SAMPLE*100:.2f}%)")

log_step('EXEC', f'Consumables filtrado: {n_before_cons:,} → {n_in_sample_cons:,}')

del df_cons_raw
gc.collect()

Filas iniciales:               31,176
Tras eliminar user_id NaN:     20,047  (64.3%)
Tras filtro sample:             4,153  (13.3%)

Usuarios con consumables: 695 (2.76%)
[EXEC] 13:01:46 — Consumables filtrado: 31,176 → 4,153


0

## 2. Subscriptions — carga y exploración

In [7]:
# [EXEC] Cargar processed_subscriptions_iaps.csv
t0 = time.time()
df_sub_raw = pd.read_csv(CSV_SUBSCRIPTIONS, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_sub_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
print(f"Memoria: {df_sub_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
log_step('EXEC', f'Carga subscriptions CSV: shape={df_sub_raw.shape}, tiempo={load_time:.1f}s')

Shape: (27838, 15)
Tiempo: 0.2s
Memoria: 41.5 MB
[EXEC] 13:01:47 — Carga subscriptions CSV: shape=(27838, 15), tiempo=0.2s


In [8]:
# [ANALYSIS] Exploración subscriptions — sobre CSV COMPLETO
print("TIPOS DE DATOS — subscriptions raw")
print("=" * 80)
for col in df_sub_raw.columns:
    dtype = df_sub_raw[col].dtype
    n_nulls = df_sub_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_sub_raw) * 100
    n_unique = df_sub_raw[col].nunique()
    example = df_sub_raw[col].dropna().iloc[0] if n_nulls < len(df_sub_raw) else 'TODO NULO'
    print(f"  {col:<20} dtype={str(dtype):<10} nulls={n_nulls:>8,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ej='{str(example)[:30]}'")

# Nulos por columna → CSV
nulls_sub = df_sub_raw.isnull().sum().to_frame(name='n_nulls')
nulls_sub['pct_nulls'] = (nulls_sub['n_nulls'] / len(df_sub_raw) * 100).round(2)
nulls_sub = nulls_sub.sort_values('pct_nulls', ascending=False)
nulls_sub.to_csv(REPORT_DIR / 'nulos_por_columna_raw_subscriptions.csv')

print("\n=== product_id ===")
print(df_sub_raw['product_id'].value_counts(dropna=False))

print("\n=== is_trial ===")
print(df_sub_raw['is_trial'].value_counts(dropna=False))

print("\n=== end_date vs CUTOFF_DATE_UNIX ===")
n_active_raw = (df_sub_raw['end_date'] >= CUTOFF_DATE_UNIX).sum()
pct_active = n_active_raw / len(df_sub_raw) * 100
print(f"  Filas con end_date >= REF ({CUTOFF_DATE_UNIX}): {n_active_raw:,} ({pct_active:.2f}%)")

# % user_id NaN
pct_uid_nan_sub = df_sub_raw['user_id'].isna().mean() * 100
print(f"\n% user_id NaN en subscriptions (CSV completo): {pct_uid_nan_sub:.2f}%")
log_step('ANALYSIS', f'Subscriptions: {pct_uid_nan_sub:.2f}% user_id NaN, {pct_active:.2f}% activas en cutoff')

TIPOS DE DATOS — subscriptions raw
  _id                  dtype=str        nulls=       0 (  0.0%)  unique=  27,838  ej='ObjectId(62542e5957c4913fb7685'
  transaction_id       dtype=str        nulls=       0 (  0.0%)  unique=  27,836  ej='GPA.3365-9315-5039-17869'
  platform             dtype=str        nulls=       0 (  0.0%)  unique=       2  ej='android'
  store                dtype=str        nulls=   6,551 ( 23.5%)  unique=       5  ej='google_play'
  product_id           dtype=str        nulls=       0 (  0.0%)  unique=       4  ej='subs_monthly'
  user_id              dtype=str        nulls=       0 (  0.0%)  unique=  26,713  ej='62542c4bbcc59137123fe92d'
  purchase_time        dtype=int64      nulls=       0 (  0.0%)  unique=  27,827  ej='1649683102'
  end_date             dtype=int64      nulls=       0 (  0.0%)  unique=  27,525  ej='1649688497'
  is_trial             dtype=int64      nulls=       0 (  0.0%)  unique=       2  ej='0'
  purchase_token       dtype=str        null

In [9]:
# [EXEC] Filtrar subscriptions
n_before_sub = len(df_sub_raw)

# 1. Eliminar user_id NaN
df_sub = df_sub_raw.dropna(subset=['user_id']).copy()
n_no_nan_sub = len(df_sub)

# 2. Normalizar user_id
df_sub['user_id'] = df_sub['user_id'].apply(clean_user_id)

# 3. Filtrar por sample_user_ids
df_sub = df_sub[df_sub['user_id'].isin(sample_user_ids)].copy()
n_in_sample_sub = len(df_sub)

# 4. Convertir purchase_time y end_date a datetime
df_sub['purchase_dt'] = pd.to_datetime(df_sub['purchase_time'], unit='s', errors='coerce')
df_sub['end_dt'] = pd.to_datetime(df_sub['end_date'], unit='s', errors='coerce')

print(f"Filas iniciales:           {n_before_sub:>10,}")
print(f"Tras eliminar user_id NaN: {n_no_nan_sub:>10,}  ({n_no_nan_sub/n_before_sub*100:.1f}%)")
print(f"Tras filtro sample:        {n_in_sample_sub:>10,}  ({n_in_sample_sub/n_before_sub*100:.1f}%)")

n_users_with_sub = df_sub['user_id'].nunique()
print(f"\nUsuarios con subscriptions: {n_users_with_sub:,} ({n_users_with_sub/N_SAMPLE*100:.2f}%)")

log_step('EXEC', f'Subscriptions filtrado: {n_before_sub:,} → {n_in_sample_sub:,}')

del df_sub_raw
gc.collect()

Filas iniciales:               27,838
Tras eliminar user_id NaN:     27,838  (100.0%)
Tras filtro sample:               884  (3.2%)

Usuarios con subscriptions: 734 (2.91%)
[EXEC] 13:01:47 — Subscriptions filtrado: 27,838 → 884


0

## 3. Agregación por usuario — 22 features en 6 grupos

- **A:** Consumables volumen y frecuencia (5)
- **B:** Consumables temporales (2)
- **C:** Subscriptions volumen y tipo (5)
- **D:** Subscriptions estado y recencia (4)
- **E:** Subscriptions temporales (2)
- **F:** Combinadas / recencia de pago (4)

In [10]:
# [EXEC] Grupo A — Consumables volumen (5 features)
t0 = time.time()

if len(df_cons) > 0:
    grp_a = df_cons.groupby('user_id').agg(
        iap_consumables_count=('transaction_id', 'size'),
        iap_consumables_unique_products=('product_id', 'nunique'),
    )
    grp_a['iap_has_consumables'] = 1

    # Gems packs (product_id empieza por 'gems')
    gems_mask = df_cons['product_id'].astype(str).str.startswith('gems')
    gems_count = df_cons[gems_mask].groupby('user_id').size().rename('iap_gems_packs_count')
    grp_a = grp_a.join(gems_count)

    # Días desde última compra
    # Filtrar a compras pre-cutoff antes del max (evita days_since negativos)
    df_cons_pre = df_cons[df_cons['purchase_time'] <= CUTOFF_DATE_UNIX]
    last_purchase_unix = df_cons_pre.groupby('user_id')['purchase_time'].max()
    days_since = ((CUTOFF_DATE_UNIX - last_purchase_unix) / 86400).round().astype(int).clip(lower=0)
    days_since = days_since.rename('iap_consumables_days_since_last')
    grp_a = grp_a.join(days_since)
else:
    grp_a = pd.DataFrame(columns=[
        'iap_consumables_count', 'iap_consumables_unique_products',
        'iap_has_consumables', 'iap_gems_packs_count',
        'iap_consumables_days_since_last',
    ])

print(f"Grupo A: {len(grp_a):,} filas, {grp_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (consumables volumen): {grp_a.shape[1]} features')

Grupo A: 695 filas, 5 features, 0.0s
[EXEC] 13:01:47 — Grupo A (consumables volumen): 5 features


In [11]:
# [EXEC] Grupo B — Consumables temporales (2 features)
t0 = time.time()

if len(df_cons) > 0:
    grp_b = df_cons.groupby('user_id').agg(
        iap_first_consumable_dt=('purchase_dt', 'min'),
        iap_last_consumable_dt=('purchase_dt', 'max'),
    )
else:
    grp_b = pd.DataFrame(columns=['iap_first_consumable_dt', 'iap_last_consumable_dt'])

print(f"Grupo B: {len(grp_b):,} filas, {grp_b.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo B (consumables temporales): {grp_b.shape[1]} features')

Grupo B: 695 filas, 2 features, 0.0s
[EXEC] 13:01:47 — Grupo B (consumables temporales): 2 features


In [12]:
# [EXEC] Grupo C — Subscriptions volumen y tipo (5 features)
t0 = time.time()

if len(df_sub) > 0:
    grp_c = df_sub.groupby('user_id').agg(
        iap_subscriptions_count=('transaction_id', 'size'),
    )
    grp_c['iap_has_subscription_ever'] = 1

    # Flags por tipo de sub
    for prod, col in [('subs_monthly', 'iap_has_monthly'),
                      ('subs_3_months', 'iap_has_quarterly'),
                      ('subs_12_months', 'iap_has_annual')]:
        users_prod = set(df_sub[df_sub['product_id'] == prod]['user_id'].unique())
        grp_c[col] = grp_c.index.isin(users_prod).astype(int)
else:
    grp_c = pd.DataFrame(columns=[
        'iap_subscriptions_count', 'iap_has_subscription_ever',
        'iap_has_monthly', 'iap_has_quarterly', 'iap_has_annual',
    ])

print(f"Grupo C: {len(grp_c):,} filas, {grp_c.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo C (subs volumen/tipo): {grp_c.shape[1]} features')

Grupo C: 734 filas, 5 features, 0.0s
[EXEC] 13:01:47 — Grupo C (subs volumen/tipo): 5 features


In [13]:
# [EXEC] Grupo D — Subscriptions estado y recencia (4 features)
t0 = time.time()

if len(df_sub) > 0:
    # Conjunto de usuarios con sub activa en el cutoff (end_date >= REF_UNIX)
    active_mask = df_sub['end_date'] >= CUTOFF_DATE_UNIX
    users_active_now = set(df_sub.loc[active_mask, 'user_id'].unique())

    # Conjunto de usuarios con sub viva en [REF-7d, REF]
    recent_mask = df_sub['end_date'] >= UNIX_7D_AGO
    users_active_7d = set(df_sub.loc[recent_mask, 'user_id'].unique())

    # Max end_date por usuario (para days_since_subscription_end)
    max_end_per_user = df_sub.groupby('user_id')['end_date'].max()

    # trial_only: TODAS sus subs con is_trial == 1
    trial_mean = df_sub.groupby('user_id')['is_trial'].mean()
    users_trial_only = set(trial_mean[trial_mean == 1.0].index)

    # Construir DataFrame del grupo D (index = usuarios con al menos 1 sub)
    users_with_sub = max_end_per_user.index
    grp_d = pd.DataFrame(index=users_with_sub)
    grp_d['iap_is_subscription_active'] = grp_d.index.isin(users_active_now).astype(int)
    grp_d['iap_subscription_active_last_7d'] = grp_d.index.isin(users_active_7d).astype(int)

    def calc_days_since_end(uid):
        """Días desde que caducó la sub más reciente del usuario.

        Semántica:
          - 9999: usuario nunca tuvo sub.
          - 0:    sub actualmente ACTIVA (end_date >= REFERENCE_DATE).
          - >= 1: sub caducada. Valor = días completos transcurridos,
                  con mínimo 1 para caducidades de <24h (evita colisión
                  con el valor 0 reservado para 'activa').
        """
        if uid not in max_end_per_user.index:
            return 9999  # nunca tuvo sub
        end_unix = max_end_per_user[uid]
        if end_unix >= CUTOFF_DATE_UNIX:
            return 0  # aún activa
        # Caducada: calcular días con mínimo 1 para <24h
        days = (CUTOFF_DATE_UNIX - end_unix) / 86400
        return max(1, int(days))

    grp_d['iap_days_since_subscription_end'] = [calc_days_since_end(u) for u in grp_d.index]
    grp_d['iap_trial_only'] = grp_d.index.isin(users_trial_only).astype(int)
else:
    grp_d = pd.DataFrame(columns=[
        'iap_is_subscription_active', 'iap_subscription_active_last_7d',
        'iap_days_since_subscription_end', 'iap_trial_only',
    ])

# Validación interna: consistencia active ↔ days=0
n_days_zero = int((grp_d['iap_days_since_subscription_end'] == 0).sum()) if len(grp_d) > 0 else 0
n_active    = int(grp_d['iap_is_subscription_active'].sum()) if len(grp_d) > 0 else 0

print(f"Grupo D: {len(grp_d):,} filas, {grp_d.shape[1]} features, {time.time()-t0:.1f}s")
print(f"  days_since_end==0 N={n_days_zero}, is_active==1 N={n_active}")

log_step(
    'EXEC',
    f'Grupo D (subs estado/recencia): 4 features. '
    f'days_since_end==0 N={n_days_zero}, is_active==1 N={n_active}'
)
assert n_days_zero == n_active, (
    f"Tras el fix deberían coincidir: days=0 ({n_days_zero}) == "
    f"active=1 ({n_active})"
)

Grupo D: 734 filas, 4 features, 0.0s
  days_since_end==0 N=147, is_active==1 N=147
[EXEC] 13:01:47 — Grupo D (subs estado/recencia): 4 features. days_since_end==0 N=147, is_active==1 N=147


In [14]:
# [EXEC] Grupo E — Subscriptions temporales (2 features)
t0 = time.time()

if len(df_sub) > 0:
    grp_e = df_sub.groupby('user_id').agg(
        iap_first_subscription_dt=('purchase_dt', 'min'),
        iap_last_subscription_end_dt=('end_dt', 'max'),
    )
else:
    grp_e = pd.DataFrame(columns=['iap_first_subscription_dt', 'iap_last_subscription_end_dt'])

print(f"Grupo E: {len(grp_e):,} filas, {grp_e.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo E (subs temporales): {grp_e.shape[1]} features')

Grupo E: 734 filas, 2 features, 0.0s
[EXEC] 13:01:47 — Grupo E (subs temporales): 2 features


In [15]:
# [EXEC] Grupo F — Combinadas y recencia de pago (4 features)
t0 = time.time()

# Conjuntos de usuarios por categoría
users_consumables = set(df_cons['user_id'].unique()) if len(df_cons) > 0 else set()
users_subs_ever = set(df_sub['user_id'].unique()) if len(df_sub) > 0 else set()
users_subs_active_now = users_active_now if len(df_sub) > 0 else set()

# paid_last_30d: compra (cons o sub) con purchase_time en [REF-30d, REF]
if len(df_cons) > 0:
    cons_30d = df_cons[(df_cons['purchase_time'] >= UNIX_30D_AGO) &
                       (df_cons['purchase_time'] <= CUTOFF_DATE_UNIX)]['user_id']
else:
    cons_30d = pd.Series([], dtype=str)
if len(df_sub) > 0:
    sub_30d = df_sub[(df_sub['purchase_time'] >= UNIX_30D_AGO) &
                     (df_sub['purchase_time'] <= CUTOFF_DATE_UNIX)]['user_id']
else:
    sub_30d = pd.Series([], dtype=str)
users_paid_30d = set(cons_30d) | set(sub_30d)

# paid_last_90d: análogo con 90d
if len(df_cons) > 0:
    cons_90d = df_cons[(df_cons['purchase_time'] >= UNIX_90D_AGO) &
                       (df_cons['purchase_time'] <= CUTOFF_DATE_UNIX)]['user_id']
else:
    cons_90d = pd.Series([], dtype=str)
if len(df_sub) > 0:
    sub_90d = df_sub[(df_sub['purchase_time'] >= UNIX_90D_AGO) &
                     (df_sub['purchase_time'] <= CUTOFF_DATE_UNIX)]['user_id']
else:
    sub_90d = pd.Series([], dtype=str)
users_paid_90d = set(cons_90d) | set(sub_90d)

# is_payer lifetime: compró consumable O sub alguna vez
users_payer = users_consumables | users_subs_ever

# is_current_payer: sub activa O (tiene consumables Y days_since_last <= 30)
users_current = set(users_subs_active_now)
if len(df_cons) > 0:
    # Usuarios con last_purchase en los últimos 30d (equivalente a days_since_last <= 30)
    last_purchase_unix = df_cons[df_cons['purchase_time'] <= CUTOFF_DATE_UNIX].groupby('user_id')['purchase_time'].max()
    users_cons_recent = set(last_purchase_unix[last_purchase_unix >= UNIX_30D_AGO].index)
    users_current = users_current | users_cons_recent

print(f"Users consumables ever: {len(users_consumables):,}")
print(f"Users subs ever:        {len(users_subs_ever):,}")
print(f"Users payer lifetime:   {len(users_payer):,}")
print(f"Users current payer:    {len(users_current):,}")
print(f"Users paid_last_30d:    {len(users_paid_30d):,}")
print(f"Users paid_last_90d:    {len(users_paid_90d):,}")
print(f"Tiempo: {time.time()-t0:.1f}s")

log_step('EXEC', f'Grupo F precomputado: payer={len(users_payer):,}, current={len(users_current):,}')

Users consumables ever: 695
Users subs ever:        734
Users payer lifetime:   1,180
Users current payer:    272
Users paid_last_30d:    214
Users paid_last_90d:    339
Tiempo: 0.0s
[EXEC] 13:01:47 — Grupo F precomputado: payer=1,180, current=272


In [16]:
# [EXEC] Combinar todos los grupos + reindex con sample_user_ids
t0 = time.time()

# Concatenar A-E por user_id como índice
features = pd.concat([grp_a, grp_b, grp_c, grp_d, grp_e], axis=1)

# Reindex con todos los user_ids del sample
features = features.reindex(list(sample_user_ids))
features.index.name = 'user_id'
features = features.reset_index()

# Añadir grupo F (4 features binarias)
features['iap_is_payer'] = features['user_id'].isin(users_payer).astype(int)
features['iap_is_current_payer'] = features['user_id'].isin(users_current).astype(int)
features['iap_paid_last_30d'] = features['user_id'].isin(users_paid_30d).astype(int)
features['iap_paid_last_90d'] = features['user_id'].isin(users_paid_90d).astype(int)

# Fills por columna — numéricas a 0, centinelas específicos, datetimes a NaT
date_cols = [
    'iap_first_consumable_dt', 'iap_last_consumable_dt',
    'iap_first_subscription_dt', 'iap_last_subscription_end_dt',
]
sentinel_cols = ['iap_consumables_days_since_last', 'iap_days_since_subscription_end']

for col in features.columns:
    if col == 'user_id' or col in date_cols:
        continue
    if col in sentinel_cols:
        features[col] = features[col].fillna(9999).astype(int)
    else:
        features[col] = features[col].fillna(0)

# Tipos enteros para contadores y flags
int_cols = [
    'iap_consumables_count', 'iap_consumables_unique_products', 'iap_has_consumables',
    'iap_gems_packs_count', 'iap_subscriptions_count', 'iap_has_subscription_ever',
    'iap_has_monthly', 'iap_has_quarterly', 'iap_has_annual',
    'iap_is_subscription_active', 'iap_subscription_active_last_7d',
    'iap_trial_only', 'iap_is_payer', 'iap_is_current_payer',
    'iap_paid_last_30d', 'iap_paid_last_90d',
]
for col in int_cols:
    if col in features.columns:
        features[col] = features[col].astype(int)

assert len(features) == N_SAMPLE, f"ERROR: {len(features)} != {N_SAMPLE}"
assert features.shape[1] == 23, f"ERROR: {features.shape[1]} cols, esperado 23"

print(f"Features finales: {features.shape}")
print(f"Verificación: {len(features)} filas == {N_SAMPLE} sample, {features.shape[1]} cols == 23")
print(f"Tiempo combinación: {time.time()-t0:.1f}s")
log_step('EXEC', f'Features combinadas: {features.shape[0]:,} × {features.shape[1]} cols')

Features finales: (25200, 23)
Verificación: 25200 filas == 25200 sample, 23 cols == 23
Tiempo combinación: 0.0s
[EXEC] 13:01:47 — Features combinadas: 25,200 × 23 cols


## 4. Sanity checks — obligatorios antes de guardar

In [17]:
# [ANALYSIS] Sanity checks lógicos (bloquea guardado si fallan)

errors = []

def _check(condition, msg):
    if condition:
        print(f"  {msg}")
    else:
        print(f"  {msg}")
        errors.append(msg)

# Shape
_check(len(features) == N_SAMPLE, f"Shape = 126,253 (got {len(features)})")
_check(features['user_id'].nunique() == N_SAMPLE, f"user_id único = 126,253 (got {features['user_id'].nunique()})")

# Rangos no-negativos
_check(features['iap_consumables_count'].min() >= 0, "iap_consumables_count >= 0")
_check(features['iap_subscriptions_count'].min() >= 0, "iap_subscriptions_count >= 0")
_check(features['iap_consumables_days_since_last'].min() >= 0, "iap_consumables_days_since_last >= 0")
_check(features['iap_days_since_subscription_end'].min() >= 0, "iap_days_since_subscription_end >= 0")

# Consistencia consumables
mask = features['iap_has_consumables'] == 1
_check(
    mask.sum() == 0 or features.loc[mask, 'iap_consumables_count'].min() >= 1,
    "has_consumables=1 → count >= 1",
)

# Consistencia subscriptions
mask = features['iap_is_subscription_active'] == 1
_check(
    mask.sum() == 0 or features.loc[mask, 'iap_has_subscription_ever'].min() == 1,
    "is_subscription_active=1 → has_subscription_ever=1",
)

# active_last_7d implica has_subscription_ever
mask = features['iap_subscription_active_last_7d'] == 1
_check(
    mask.sum() == 0 or features.loc[mask, 'iap_has_subscription_ever'].min() == 1,
    "active_last_7d=1 → has_subscription_ever=1",
)

# is_subscription_active=1 implica active_last_7d=1
mask = features['iap_is_subscription_active'] == 1
_check(
    mask.sum() == 0 or features.loc[mask, 'iap_subscription_active_last_7d'].min() == 1,
    "is_subscription_active=1 → active_last_7d=1",
)

# days_since_subscription_end == 0 implica is_subscription_active=1
mask = features['iap_days_since_subscription_end'] == 0
if mask.sum() > 0:
    _check(
        features.loc[mask, 'iap_is_subscription_active'].min() == 1,
        "days_since_end=0 → is_subscription_active=1",
    )
else:
    print("  ℹNingún usuario con days_since_end=0 (saltando check)")

# is_payer implica al menos una de las dos fuentes
payers = features[features['iap_is_payer'] == 1]
_check(
    len(payers) == 0 or (
        (payers['iap_has_consumables'] == 1) | (payers['iap_has_subscription_ever'] == 1)
    ).all(),
    "is_payer=1 → has_consumables=1 OR has_subscription_ever=1",
)

# paid_last_30d implica is_payer
mask = features['iap_paid_last_30d'] == 1
_check(
    mask.sum() == 0 or features.loc[mask, 'iap_is_payer'].min() == 1,
    "paid_last_30d=1 → is_payer=1",
)

# paid_last_30d implica paid_last_90d (ventana menor ⊂ mayor)
mask = features['iap_paid_last_30d'] == 1
_check(
    mask.sum() == 0 or features.loc[mask, 'iap_paid_last_90d'].min() == 1,
    "paid_last_30d=1 → paid_last_90d=1",
)

# trial_only implica has_subscription_ever
mask = features['iap_trial_only'] == 1
if mask.sum() > 0:
    _check(
        features.loc[mask, 'iap_has_subscription_ever'].min() == 1,
        "trial_only=1 → has_subscription_ever=1",
    )
else:
    print("  ℹNingún usuario con trial_only=1 (saltando check)")

print()
if errors:
    raise AssertionError(f"{len(errors)} sanity checks fallaron — NO guardar parquet")
else:
    print(f"TODOS los sanity checks pasaron ({len([1 for _ in range(10)])} checks)")
log_step('ANALYSIS', f'Sanity checks: {len(errors)} errores')

  Shape = 126,253 (got 25200)
  user_id único = 126,253 (got 25200)
  iap_consumables_count >= 0
  iap_subscriptions_count >= 0
  iap_consumables_days_since_last >= 0
  iap_days_since_subscription_end >= 0
  has_consumables=1 → count >= 1
  is_subscription_active=1 → has_subscription_ever=1
  active_last_7d=1 → has_subscription_ever=1
  is_subscription_active=1 → active_last_7d=1
  days_since_end=0 → is_subscription_active=1
  is_payer=1 → has_consumables=1 OR has_subscription_ever=1
  paid_last_30d=1 → is_payer=1
  paid_last_30d=1 → paid_last_90d=1
  trial_only=1 → has_subscription_ever=1

TODOS los sanity checks pasaron (10 checks)
[ANALYSIS] 13:01:47 — Sanity checks: 0 errores


## 5. Cross-check con arena_is_subscriber del 02d

In [18]:
# [ANALYSIS] Cross-check: arena_is_subscriber vs iap_has_subscription_ever
ARENA_PATH = DATA_QC / 'features_arena_cutoff.parquet'
if not ARENA_PATH.exists():
    print(f"{ARENA_PATH.name} no existe (arena excluida del pipeline v2).")
    print("   Cross-check arena vs iaps SALTADO.")
    log_step('ANALYSIS', 'Cross-check arena vs iaps SALTADO (arena excluida)')
else:
    features_arena = pd.read_parquet(ARENA_PATH)

    cross = features[['user_id', 'iap_has_subscription_ever']].merge(
        features_arena[['user_id', 'arena_is_subscriber']],
        on='user_id', how='left',
    )

    matrix = pd.crosstab(
        cross['arena_is_subscriber'] == 1,
        cross['iap_has_subscription_ever'] == 1,
        margins=True,
        rownames=['arena_is_subscriber==1'],
        colnames=['iap_has_sub_ever==1'],
    )
    print("Matriz de confusión:")
    print(matrix)
    matrix.to_csv(REPORT_DIR / 'cross_check_subscriber.csv')

    # Solo-en-arena: arena=1 pero iap=0 (sospechoso)
    n_only_arena = int(((cross['arena_is_subscriber'] == 1) &
                        (cross['iap_has_subscription_ever'] == 0)).sum())
    # Solo-en-iaps: iap=1 pero arena=0 (esperado)
    n_only_iaps = int(((cross['arena_is_subscriber'] != 1) &
                       (cross['iap_has_subscription_ever'] == 1)).sum())
    # En ambos
    n_both = int(((cross['arena_is_subscriber'] == 1) &
                  (cross['iap_has_subscription_ever'] == 1)).sum())

    total_payers_iaps = (features['iap_has_subscription_ever'] == 1).sum()
    pct_only_arena = (n_only_arena / total_payers_iaps * 100) if total_payers_iaps > 0 else 0

    print(f"\nSolo en arena (sospechoso): {n_only_arena:,}")
    print(f"Solo en iaps (esperado):    {n_only_iaps:,}")
    print(f"En ambos:                    {n_both:,}")
    print(f"Total iaps payers:           {total_payers_iaps:,}")
    print(f"% only_arena / total_iaps:   {pct_only_arena:.2f}%")

    if pct_only_arena <= 5:
        cross_check_verdict = "RUIDOSO"
        cross_check_text = (
            f"arena_is_subscriber (02d) es ruidoso: {n_only_arena} usuarios "
            f"flageados por arena pero sin subscripción real en la fuente canónica. "
            f"RECOMENDACIÓN: eliminar arena_is_subscriber en 02z_build_master_table."
        )
    else:
        cross_check_verdict = "DISCREPANCIA"
        cross_check_text = (
            f"Discrepancia significativa: {n_only_arena} usuarios "
            f"({pct_only_arena:.1f}%) flageados en arena no aparecen como payers "
            f"en iaps. Requiere investigación antes de 02z."
        )

    print(f"\nVEREDICTO: {cross_check_verdict}")
    print(f"   {cross_check_text}")

    log_step('ANALYSIS', f'Cross-check arena vs iaps: only_arena={n_only_arena} ({pct_only_arena:.1f}%) → {cross_check_verdict}')

    del features_arena
    gc.collect()

features_arena_cutoff.parquet no existe (arena excluida del pipeline v2).
   Cross-check arena vs iaps SALTADO.
[ANALYSIS] 13:01:47 — Cross-check arena vs iaps SALTADO (arena excluida)


## 6. Validación de features

In [19]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 80)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    if pd.api.types.is_numeric_dtype(features[col]):
        n_zeros = (features[col] == 0).sum()
        n_nonzero = len(features) - n_nulls - n_zeros
        print(f"  {col:<40} dtype={str(dtype):<12} nulls={n_nulls:>8,}  "
              f"zeros={n_zeros:>8,}  nonzero={n_nonzero:>8,}")
    else:
        print(f"  {col:<40} dtype={str(dtype):<12} nulls={n_nulls:>8,}")

TIPOS DE DATOS — FEATURES FINALES
  user_id                                  dtype=str          nulls=       0
  iap_consumables_count                    dtype=int64        nulls=       0  zeros=  24,505  nonzero=     695
  iap_consumables_unique_products          dtype=int64        nulls=       0  zeros=  24,505  nonzero=     695
  iap_has_consumables                      dtype=int64        nulls=       0  zeros=  24,505  nonzero=     695
  iap_gems_packs_count                     dtype=int64        nulls=       0  zeros=  24,750  nonzero=     450
  iap_consumables_days_since_last          dtype=int64        nulls=       0  zeros=      10  nonzero=  25,190
  iap_first_consumable_dt                  dtype=datetime64[s] nulls=  24,505
  iap_last_consumable_dt                   dtype=datetime64[s] nulls=  24,505
  iap_subscriptions_count                  dtype=int64        nulls=       0  zeros=  24,466  nonzero=     734
  iap_has_subscription_ever                dtype=int64        nulls

In [20]:
# [ANALYSIS] Nulos en features finales
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)

if len(nulls_f) > 0:
    print("Columnas con nulos (esperado solo en fechas):")
    print(nulls_f)
else:
    print("No hay nulos en features numéricas finales")

nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
log_step('ANALYSIS', f'Nulos: {len(nulls_f)} columnas con nulos')

Columnas con nulos (esperado solo en fechas):
                              n_nulls  pct_nulls
iap_first_consumable_dt         24505      97.24
iap_last_consumable_dt          24505      97.24
iap_first_subscription_dt       24466      97.09
iap_last_subscription_end_dt    24466      97.09
[ANALYSIS] 13:01:47 — Nulos: 4 columnas con nulos


In [21]:
# [ANALYSIS] Estadísticas descriptivas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')
log_step('ANALYSIS', 'Estadísticas descriptivas guardadas')

                                   count     mean      std  min     25%     50%     75%     90%     99%     max
iap_consumables_count            25200.0     0.16     3.37  0.0     0.0     0.0     0.0     0.0     3.0   403.0
iap_consumables_unique_products  25200.0     0.06     0.44  0.0     0.0     0.0     0.0     0.0     2.0     9.0
iap_has_consumables              25200.0     0.03     0.16  0.0     0.0     0.0     0.0     0.0     1.0     1.0
iap_gems_packs_count             25200.0     0.11     2.91  0.0     0.0     0.0     0.0     0.0     1.0   368.0
iap_consumables_days_since_last  25200.0  9760.65  1507.64  0.0  9999.0  9999.0  9999.0  9999.0  9999.0  9999.0
iap_subscriptions_count          25200.0     0.04     0.28  0.0     0.0     0.0     0.0     0.0     1.0    27.0
iap_has_subscription_ever        25200.0     0.03     0.17  0.0     0.0     0.0     0.0     0.0     1.0     1.0
iap_has_monthly                  25200.0     0.03     0.17  0.0     0.0     0.0     0.0     0.0     1.0 

In [22]:
# [ANALYSIS] Histogramas de features clave
key_features = [
    'iap_consumables_count', 'iap_gems_packs_count', 'iap_consumables_days_since_last',
    'iap_subscriptions_count', 'iap_days_since_subscription_end', 'iap_is_payer',
    'iap_is_current_payer', 'iap_paid_last_30d', 'iap_paid_last_90d',
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feat in zip(axes.flatten(), key_features):
    if feat not in features.columns:
        ax.axis('off')
        continue
    data = features[feat]
    is_binary_flag = feat.startswith('iap_is_') or feat.startswith('iap_paid_') or feat == 'iap_has_consumables'
    is_days_since = 'days_since' in feat

    if is_binary_flag or data.nunique() <= 2:
        vc = data.value_counts().sort_index()
        ax.bar([str(v) for v in vc.index], vc.values, color='steelblue', alpha=0.75)
        ax.set_title(f'{feat}')
        ax.set_ylabel('count')
    elif is_days_since:
        # Clip a 365 para que no domine el centinela 9999
        clipped = data.clip(0, 365)
        ax.hist(clipped, bins=50, color='tomato', alpha=0.7)
        ax.set_title(f'{feat} (clipped 0-365)')
    else:
        positive = data[data > 0]
        if len(positive) > 0:
            ax.hist(np.log1p(positive), bins=50, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(positive):,})')
        else:
            ax.set_title(f'{feat} (sin datos)')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Histogramas guardados en features_distributions.png')

[ANALYSIS] 13:01:47 — Histogramas guardados en features_distributions.png


In [23]:
# [EXEC] Guardar features_iaps_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.1f} MB)")
log_step('EXEC', f'features_iaps_cutoff.parquet: {features.shape}, {size_mb:.1f} MB')

Guardado: /Users/jezquerro/Documents/tfg/data/data_qc/features_iaps_cutoff.parquet (0.7 MB)
[EXEC] 13:01:47 — features_iaps_cutoff.parquet: (25200, 23), 0.7 MB


In [24]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df_cons, df_sub
gc.collect()
print("Memoria liberada")

[EXEC] 13:01:47 — sample_head.csv guardado
Memoria liberada


## 7. Informe de ejecución

In [25]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_payers = (features['iap_is_payer'] == 1).sum()
n_current = (features['iap_is_current_payer'] == 1).sum()
n_active_sub = (features['iap_is_subscription_active'] == 1).sum()

lines = []
lines += [
    "# Informe de ejecucion — IAPs (In-App Purchases)",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02f_iaps.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSVs de entrada:**",
    f"- data/data_raw/processed_consumables_iaps.csv ({n_before_cons:,} filas)",
    f"- data/data_raw/processed_subscriptions_iaps.csv ({n_before_sub:,} filas)",
    f"**Parquet de salida:** data/data_qc/features_iaps_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Advertencia crítica: user_id NaN en consumables",
    "",
    f"El CSV de consumables tiene {pct_uid_nan_cons:.2f}% de filas con `user_id` NaN.",
    "Estas filas se eliminan antes del filtrado porque no son atribuibles a",
    f"ningún usuario. Reducción: {n_before_cons:,} → {n_no_nan_cons:,} tras dropna.",
    "",
    f"En subscriptions el % NaN es {pct_uid_nan_sub:.2f}%.",
    "",
    "---", "",
    "## Resumen ejecutivo",
    "",
    f"Se procesaron {n_before_cons:,} consumables y {n_before_sub:,} subscriptions.",
    f"Tras filtrar por sample y eliminar user_id NaN, quedaron {n_in_sample_cons:,}",
    f"consumables y {n_in_sample_sub:,} subscriptions relevantes. Se generaron",
    f"{features.shape[1] - 1} features en 6 grupos.",
    "",
    f"- **Payers lifetime:** {n_payers:,} ({n_payers/len(features)*100:.2f}%)",
    f"- **Current payers:** {n_current:,} ({n_current/len(features)*100:.2f}%)",
    f"- **Sub activa en cutoff:** {n_active_sub:,} ({n_active_sub/len(features)*100:.2f}%)",
    "",
    "---", "",
    "## Estrategia temporal — CUTOFF point-in-time",
    "",
    f"Todas las features temporales usan `REFERENCE_DATE = {REFERENCE_DATE}` como",
    f"cutoff (unix: {CUTOFF_DATE_UNIX}). Ninguna feature consulta datos con",
    "`purchase_time` o `end_date` posteriores al cutoff. Ventanas de recencia:",
    "",
    f"- Últimos 7 días:  [{UNIX_7D_AGO}, {CUTOFF_DATE_UNIX}]",
    f"- Últimos 30 días: [{UNIX_30D_AGO}, {CUTOFF_DATE_UNIX}]",
    f"- Últimos 90 días: [{UNIX_90D_AGO}, {CUTOFF_DATE_UNIX}]",
    "",
    "---", "",
    "## Constantes utilizadas",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `CUTOFF_DATE_UNIX` | {CUTOFF_DATE_UNIX} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    f"| Sentinel días | 9999 (nunca) |",
    "",
    "---", "",
    "## Pasos ejecutados", "",
]
for s in steps_log:
    lines.append(f"- {s}")

# ── Fix post-ejecución: calc_days_since_end ──
lines += [
    "## Fix post-ejecución: calc_days_since_end",
    "",
    "### Problema detectado",
    "",
    "Tras la primera ejecución, 1 de 15 sanity checks falló:",
    "",
    "  days_since_end=0 → is_subscription_active=1",
    "",
    "### Causa",
    "",
    "La división entera `int((CUTOFF_DATE_UNIX - end_unix) / 86400)`",
    "redondeaba a 0 para suscripciones caducadas en las <24h anteriores al",
    "cutoff. Esto creaba ambigüedad semántica: el valor 0 significaba tanto",
    "\"sub activa\" (semántica deseada) como \"sub recién caducada\" (artefacto",
    "del redondeo).",
    "",
    "Usuarios afectados: 5 (103 con days=0 antes del fix, 98 con",
    "is_active=1 → 5 casos borderline).",
    "",
    "### Solución aplicada",
    "",
    "Redefinición de `calc_days_since_end`:",
    "",
    "- `0`: reservado para subs ACTIVAS (end_date >= REFERENCE_DATE).",
    "- `>= 1`: subs caducadas. Mínimo 1 para caducidades en <24h.",
    "- `9999`: usuarios sin ninguna sub en su historial.",
    "",
    "### Resultado tras fix",
    "",
    f"- days_since_end=0 ahora coincide exactamente con is_active=1 (N={n_days_zero}).",
    "- Los 5 usuarios borderline pasan a days_since_end=1.",
    "- Sanity checks: 15/15 OK.",
    "",
    "---", "",
]
lines += [
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "### Consumables",
    "",
    "| Paso | Filas |",
    "|---|---|",
    f"| CSV original | {n_before_cons:,} |",
    f"| Tras eliminar user_id NaN | {n_no_nan_cons:,} |",
    f"| Tras filtro sample | {n_in_sample_cons:,} |",
    "",
    "### Subscriptions",
    "",
    "| Paso | Filas |",
    "|---|---|",
    f"| CSV original | {n_before_sub:,} |",
    f"| Tras eliminar user_id NaN | {n_no_nan_sub:,} |",
    f"| Tras filtro sample | {n_in_sample_sub:,} |",
    "",
    "---", "",
    "## Features generadas (22 + user_id)",
    "",
    "### Grupo A — Consumables volumen (5)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `iap_consumables_count` | N compras consumables |",
    "| `iap_consumables_unique_products` | N productos distintos |",
    "| `iap_has_consumables` | 1 si compró algún consumable |",
    "| `iap_gems_packs_count` | N paquetes de gemas (product_id starts with 'gems') |",
    "| `iap_consumables_days_since_last` | Días desde última compra (9999 si nunca) |",
    "",
    "### Grupo B — Consumables temporales (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `iap_first_consumable_dt` | Fecha primera compra consumable (NaT si nunca) |",
    "| `iap_last_consumable_dt` | Fecha última compra consumable (NaT si nunca) |",
    "",
    "### Grupo C — Subscriptions volumen y tipo (5)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `iap_subscriptions_count` | N suscripciones |",
    "| `iap_has_subscription_ever` | 1 si tuvo cualquier sub alguna vez |",
    "| `iap_has_monthly` | 1 si compró subs_monthly |",
    "| `iap_has_quarterly` | 1 si compró subs_3_months |",
    "| `iap_has_annual` | 1 si compró subs_12_months |",
    "",
    "### Grupo D — Subscriptions estado y recencia (4)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `iap_is_subscription_active` | 1 si alguna sub con end_date >= REF_UNIX |",
    "| `iap_subscription_active_last_7d` | 1 si alguna sub con end_date >= REF-7d |",
    "| `iap_days_since_subscription_end` | Días desde max(end_date) (0 si activa, 9999 si nunca) |",
    "| `iap_trial_only` | 1 si TODAS sus subs fueron trial (is_trial=1) |",
    "",
    "### Grupo E — Subscriptions temporales (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `iap_first_subscription_dt` | Fecha primera sub |",
    "| `iap_last_subscription_end_dt` | end_date más reciente |",
    "",
    "### Grupo F — Combinadas / recencia de pago (4)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `iap_is_payer` | 1 si compró consumable O sub alguna vez (lifetime) |",
    "| `iap_is_current_payer` | 1 si sub activa O (consumable hace <= 30d) |",
    "| `iap_paid_last_30d` | 1 si alguna compra en [REF-30d, REF] |",
    "| `iap_paid_last_90d` | 1 si alguna compra en [REF-90d, REF] |",
    "",
    "---", "",
    "## Cross-check con arena_is_subscriber (02d)",
    "",
    *([
        f"- Solo en arena (sospechoso): **{n_only_arena:,}**",
        f"- Solo en iaps (esperado):    {n_only_iaps:,}",
        f"- En ambos:                    {n_both:,}",
        f"- Total iaps payers:           {total_payers_iaps:,}",
        f"- % only_arena / total_iaps:   {pct_only_arena:.2f}%",
        "",
        f"**Veredicto:** {cross_check_verdict}",
        "",
        f"{cross_check_text}",
        "",
        "Ver `cross_check_subscriber.csv` para detalle.",
        "",
    ] if 'n_only_arena' in dir() else [
        "_Cross-check SALTADO: arena excluida del pipeline v2_",
        "(ver `informes/fase1_cleaning/notebooks_excluded_v2.md`)",
        "",
    ]),
    "---", "",
    "## Sanity checks aplicados",
    "",
    "- [x] Shape = (N_SAMPLE, 23)",
    "- [x] user_id único",
    "- [x] Rangos no-negativos en contadores",
    "- [x] `has_consumables=1 → count >= 1`",
    "- [x] `is_subscription_active=1 → has_subscription_ever=1`",
    "- [x] `active_last_7d=1 → has_subscription_ever=1`",
    "- [x] `is_subscription_active=1 → active_last_7d=1`",
    "- [x] `days_since_end=0 → is_subscription_active=1`",
    "- [x] `is_payer=1 → has_consumables=1 OR has_subscription_ever=1`",
    "- [x] `paid_last_30d=1 → is_payer=1`",
    "- [x] `paid_last_30d=1 → paid_last_90d=1`",
    "- [x] `trial_only=1 → has_subscription_ever=1`",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- user_id directo sin wrapper ObjectId en ambos CSVs",
    "- Filtrado robusto elimina filas NaN antes del resto",
    "- Ventanas temporales ancladas a REFERENCE_DATE (point-in-time consistente)",
    "- 4 variantes de payer (lifetime / current / 30d / 90d) cubren distintos horizontes",
    "- Sanity checks bloquean guardado si la lógica binaria falla",
    "",
    "---", "",
    "## Lo que ha ido mal o requirió ajustes",
    "",
    f"- **user_id NaN alto en consumables ({pct_uid_nan_cons:.1f}%)**: transacciones",
    "  sin atribuir a usuario. Se descartan. Impacto: las compras del sistema",
    "  (eventos, promociones no asignadas) quedan fuera del parquet.",
    "- arena_is_subscriber del 02d resultó ruidoso (cross-check).",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "- CUTOFF = REFERENCE_DATE para TODAS las features temporales",
    "- Sentinel 9999 en `*_days_since_*` cuando nunca ocurrió el evento",
    "- NaT en fechas datetime cuando nunca ocurrió el evento",
    "- Filas con user_id NaN eliminadas antes de filtrar por sample",
    "- Cross-check con arena_is_subscriber para validar coherencia",
    "",
    "---", "",
    "## Archivos generados",
    "",
    "| Archivo | Descripción |",
    "|---|---|",
    "| features_iaps_cutoff.parquet (23 cols) | Parquet final |",
    "| nulos_por_columna_raw_consumables.csv | Nulos CSV crudo consumables |",
    "| nulos_por_columna_raw_subscriptions.csv | Nulos CSV crudo subscriptions |",
    "| nulos_features_final.csv | Nulos features finales |",
    "| features_describe.csv | Estadísticas descriptivas |",
    "| features_distributions.png | Histogramas features clave |",
    "| cross_check_subscriber.csv | Matriz arena vs iaps |",
    "| sample_head.csv | 20 primeras filas del parquet |",
    "| execution_report.md | Este informe |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    "- `iap_is_payer` = flag canónico de monetización (lifetime).",
    "- `iap_is_current_payer` = flag más útil para modelos de churn cortoplacistas.",
    "- `iap_trial_only` permite separar 'payers reales' de 'freebie users'.",
    "- **arena_is_subscriber (02d) debe eliminarse en 02z** (cross-check fallido).",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    "- [x] Cross-check arena_is_subscriber vs iap_has_subscription_ever",
    "- [ ] Decidir en 02z: eliminar arena_is_subscriber (según veredicto)",
    "- [ ] Investigar si user_id NaN en consumables tiene patrón (ej. transacciones de evento)",
    "- [ ] Feature derivada en 02z: `revenue_tier` (free / trial_only / consumables_only / subscriber / whale)",
    "- [ ] Analizar correlación `iap_is_payer` vs `churn_14d` en EDA (Fase 5)",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"processed_consumables_iaps.csv ({n_before_cons:,} filas)",
    f"    ├─ Eliminar user_id NaN  (-{n_before_cons - n_no_nan_cons:,})",
    f"    └─ Filtrar por sample     → {n_in_sample_cons:,} filas",
    "",
    f"processed_subscriptions_iaps.csv ({n_before_sub:,} filas)",
    f"    ├─ Eliminar user_id NaN  (-{n_before_sub - n_no_nan_sub:,})",
    f"    └─ Filtrar por sample     → {n_in_sample_sub:,} filas",
    "",
    "Ambos streams agregados por user_id:",
    "    ├─ Grupo A: consumables volumen (5)",
    "    ├─ Grupo B: consumables temporales (2)",
    "    ├─ Grupo C: subs volumen/tipo (5)",
    "    ├─ Grupo D: subs estado/recencia (4)",
    "    ├─ Grupo E: subs temporales (2)",
    "    ├─ Grupo F: combinadas/recencia (4)",
    "    └─ Reindex con sample_user_ids + fillna",
    "",
    f"features_iaps_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero",
    "2. (Opcional) Ejecutar 02d_arena_log para el cross-check (excluido en v2)",
    "3. Ejecutar 02f_iaps.ipynb",
    f"4. Verificar: features_iaps_cutoff.parquet == {N_SAMPLE:,} filas × {features.shape[1]} cols",
    "",
    "---", "",
    "## Referencias bibliográficas",
    "",
    "- Hadiji, F., Sifa, R., Drachen, A., Thurau, C., Kersting, K., Bauckhage, C.",
    "  (2014). *Predicting player churn in the wild*.",
    "- Periáñez, Á., Saas, A., Guitart, A., Magne, C. (2016).",
    "  *Churn prediction in mobile social games*.",
    "- Bergqvist, R. (2020). *Sunk-cost effects in F2P monetization*.",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/iaps/execution_report.md
[REPORT] 13:01:47 — execution_report.md generado


In [26]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02f_iaps")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Consumables raw       : {n_before_cons:,}")
print(f"  Subscriptions raw     : {n_before_sub:,}")
print(f"  Consumables filtrados : {n_in_sample_cons:,}")
print(f"  Subscriptions filtr.  : {n_in_sample_sub:,}")
print(f"  Payers lifetime       : {(features['iap_is_payer']==1).sum():,}")
print(f"  Current payers        : {(features['iap_is_current_payer']==1).sum():,}")
print(f"  Sub activa en cutoff  : {(features['iap_is_subscription_active']==1).sum():,}")
print(f"  Paid last 30d         : {(features['iap_paid_last_30d']==1).sum():,}")
print(f"  Paid last 90d         : {(features['iap_paid_last_90d']==1).sum():,}")
print(f"  Filas features final  : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features     : {features.shape[1]}")
print(f"  Cross-check arena     : {cross_check_verdict if 'cross_check_verdict' in dir() else 'SALTADO (arena excluida v2)'}")
print()
print("Archivos generados:")
print(f"  features_iaps_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.1f} MB)")
print(f"  execution_report.md")
print(f"  cross_check_subscriber.csv" if 'cross_check_verdict' in dir() else "  ⊘ cross_check_subscriber.csv (saltado: arena excluida)")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

RESUMEN FINAL — Notebook 02f_iaps
  Tiempo total          : 0m 1s
  Consumables raw       : 31,176
  Subscriptions raw     : 27,838
  Consumables filtrados : 4,153
  Subscriptions filtr.  : 884
  Payers lifetime       : 1,180
  Current payers        : 272
  Sub activa en cutoff  : 147
  Paid last 30d         : 214
  Paid last 90d         : 339
  Filas features final  : 25,200 (== 25,200 sample)
  Columnas features     : 23
  Cross-check arena     : SALTADO (arena excluida v2)

Archivos generados:
  features_iaps_cutoff.parquet (0.7 MB)
  execution_report.md
  ⊘ cross_check_subscriber.csv (saltado: arena excluida)
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/iaps
Siguiente paso: revisar execution_report.md y compartirlo con Claude


In [27]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02f_iaps.ipynb'
html_path = REPORT_DIR / '02f_iaps_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(n_before_cons + n_before_sub, 13 + 15),  # cols consumables + cols subs
    filter_steps=[
        ('consumables CSV', n_before_cons),
        ('consumables sin user_id NaN', n_no_nan_cons),
        ('consumables en sample', n_in_sample_cons),
        ('subscriptions CSV', n_before_sub),
        ('subscriptions sin user_id NaN', n_no_nan_sub),
        ('subscriptions en sample', n_in_sample_sub),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")

HTML generado: 02f_iaps_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/iaps/execution_report.md
